# Lesson 17 — Authentication & RBAC

## What you will learn
- **JWT token** validation before any graph execution
- **Role-Based Access Control (RBAC)**: admin, manager, user roles
- How to inject `user_role` into LangGraph state for routing decisions
- **Permission-gated nodes**: some nodes only run for specific roles
- Audit logging: who ran what and when

## Architecture
```
HTTP Request
   ↓
JWT Middleware → decode token → extract {user_id, tenant_id, role}
   ↓
Graph invoke with state = {..., user_role: "manager"}
   ↓
Router → checks role → allows/denies specific nodes
```

## RBAC Matrix
```
Role     │ read_data │ write_data │ delete_data │ admin_panel
─────────┼───────────┼────────────┼─────────────┼────────────
user     │    ✅     │    ❌      │     ❌      │    ❌
manager  │    ✅     │    ✅      │     ❌      │    ❌
admin    │    ✅     │    ✅      │     ✅      │    ✅
```

In [ ]:
# !pip install langgraph pyjwt

## Step 1 — JWT Token Handling

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
from config import get_ollama_model

import jwt
import time
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

JWT_SECRET = "lesson17-demo-secret"  # In production: load from env
JWT_ALGORITHM = "HS256"

def create_token(user_id: str, tenant_id: str, role: str, expires_in: int = 3600) -> str:
    payload = {
        "sub": user_id,
        "tenant_id": tenant_id,
        "role": role,
        "exp": int(time.time()) + expires_in,
        "iat": int(time.time()),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)

def verify_token(token: str) -> dict:
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
        return {"valid": True, "payload": payload}
    except jwt.ExpiredSignatureError:
        return {"valid": False, "error": "Token expired"}
    except jwt.InvalidTokenError as e:
        return {"valid": False, "error": str(e)}

# Demo tokens
tokens = {
    "alice_user":    create_token("alice",   "acme", "user"),
    "bob_manager":   create_token("bob",     "acme", "manager"),
    "carol_admin":   create_token("carol",   "acme", "admin"),
}

for name, token in tokens.items():
    result = verify_token(token)
    print(f"{name}: valid={result['valid']} | role={result['payload']['role']}")

## Step 2 — State with Auth Context

In [ ]:
class SecureState(TypedDict):
    messages:  Annotated[list, add_messages]
    user_id:   str
    tenant_id: str
    role:      str    # "user" | "manager" | "admin"
    action:    str    # what the user wants to do
    result:    str    # output
    denied:    bool   # True if access was denied

## Step 3 — Permission-Gated Nodes

In [ ]:
ROLE_PERMISSIONS = {
    "user":    {"read_data"},
    "manager": {"read_data", "write_data"},
    "admin":   {"read_data", "write_data", "delete_data", "admin_panel"},
}

def has_permission(role: str, required: str) -> bool:
    return required in ROLE_PERMISSIONS.get(role, set())

def auth_gate_node(state: SecureState) -> dict:
    """Check if role has permission for the requested action."""
    action = state["action"]
    role = state["role"]
    
    # Map action keywords to permissions
    if "delete" in action.lower() or "remove" in action.lower():
        required = "delete_data"
    elif "admin" in action.lower() or "system" in action.lower():
        required = "admin_panel"
    elif "write" in action.lower() or "create" in action.lower() or "update" in action.lower():
        required = "write_data"
    else:
        required = "read_data"
    
    if not has_permission(role, required):
        print(f"[auth_gate] DENIED: role={role} needs {required}")
        return {"denied": True, "result": f"Access denied: role '{role}' cannot '{required}'"}
    
    print(f"[auth_gate] ALLOWED: role={role} has {required}")
    return {"denied": False}

def execute_action_node(state: SecureState) -> dict:
    """Execute the action — only reached if auth passed."""
    result = f"[{state['role'].upper()}] '{state['action']}' executed by {state['user_id']} (tenant: {state['tenant_id']})"
    print(f"[execute] {result}")
    # Audit log
    print(f"  AUDIT: user={state['user_id']} | action={state['action']} | role={state['role']}")
    return {"result": result}

def route_after_auth(state: SecureState) -> str:
    return "denied" if state["denied"] else "execute"

def access_denied_node(state: SecureState) -> dict:
    return {"messages": [AIMessage(content=state["result"])]}

## Step 4 — Build and Test the Secure Graph

In [ ]:
builder = StateGraph(SecureState)
builder.add_node("auth_gate", auth_gate_node)
builder.add_node("execute",   execute_action_node)
builder.add_node("denied",    access_denied_node)
builder.add_edge(START, "auth_gate")
builder.add_conditional_edges("auth_gate", route_after_auth, {"execute": "execute", "denied": "denied"})
builder.add_edge("execute", END)
builder.add_edge("denied",  END)

graph = builder.compile()

def run_with_jwt(token: str, action: str):
    """Decode JWT, extract claims, inject into state."""
    auth = verify_token(token)
    if not auth["valid"]:
        print(f"  REJECTED: {auth['error']}")
        return
    p = auth["payload"]
    print(f"\n{'='*60}")
    print(f"User: {p['sub']} | Role: {p['role']} | Action: {action}")
    result = graph.invoke({
        "messages": [HumanMessage(content=action)],
        "user_id": p["sub"], "tenant_id": p["tenant_id"],
        "role": p["role"], "action": action,
        "result": "", "denied": False,
    })
    print(f"Result: {result['result']}")

# Test all 3 roles with various actions
run_with_jwt(tokens["alice_user"],  "read user list")
run_with_jwt(tokens["alice_user"],  "delete user record")    # should be denied
run_with_jwt(tokens["bob_manager"], "create new report")
run_with_jwt(tokens["bob_manager"], "admin system config")   # should be denied
run_with_jwt(tokens["carol_admin"], "delete old logs")
run_with_jwt(tokens["carol_admin"], "admin panel access")

## Key Takeaways

| Pattern | Implementation |
|---------|---------------|
| JWT decode | `jwt.decode(token, secret, algorithms=[...])` |
| Role injection | Decode JWT → inject `role` into state before `invoke()` |
| Auth gate node | First node checks role vs required permission |
| Conditional routing | `denied=True` → access denied node |
| Audit log | Log `user_id + action + role` in execute node |

## 🏋️ Exercise
1. Add a `superadmin` role that bypasses all permission checks
2. Add token expiry: create a token with `expires_in=1` (1 second), wait 2 seconds, try to use it
3. Add an `audit_log: list` field to state — append every action attempt (including denied ones)

In [ ]:
# Your exercise solution here
